## **Ensemble LLM: 5 Modelos via Groq — Weighted Voting**

**Tarefa 3 do Trabalho Prático — Aprendizagem Profunda**

Ensemble de 5 LLMs com weighted voting por F1/classe:
1. **GPT-OSS 120B** (OpenAI open-source)
2. **Llama 3.3 70B Versatile** (Meta)
3. **Kimi K2 Instruct** (Moonshot AI)
4. **Qwen3 32B** (Alibaba)
5. **Llama 4 Scout 17B** (Meta)

Rotação automática de 4 API keys Groq para evitar rate limits.

- **Few-shot examples:** textos revelados da submissão 1 (100 textos)
- **Teste:** dataset-exemplos do professor (125 textos)
- **N_PER_CLASS:** 30 exemplos por classe no prompt

### **1. Setup e Dados**

In [ ]:
# !pip install groq

import pandas as pd
import numpy as np
import time
import json
from groq import Groq
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

sns.set_style('whitegrid')

LABELS = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']

# Carregar dados
df_support = pd.read_csv('../database/dataset-test.csv', sep=';')
df_support.columns = df_support.columns.str.strip().str.lower()

df_test = pd.read_csv('../database/dataset-subm2-labels.csv', sep=';')
df_test.columns = df_test.columns.str.strip().str.lower()

print(f'Few-shot examples: {len(df_support)} textos')
print(f'  Distribuição: {df_support["label"].value_counts().to_dict()}')
print(f'\nTeste: {len(df_test)} textos')
print(f'  Distribuição: {df_test["label"].value_counts().to_dict()}')

### **2. Groq: Keys e Modelos**

In [ ]:
# ============================================================
# API KEYS GROQ (rotação automática)
# ============================================================
GROQ_KEYS = [
    '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',
    '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',
    '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',
    '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',
]

groq_clients = [Groq(api_key=key) for key in GROQ_KEYS]
groq_idx = 0  # Índice global para rotação

# Modelos a usar no ensemble
MODELS = {
    'GPT-OSS-120B':    'openai/gpt-oss-120b',
    'Llama3.3-70B':    'llama-3.3-70b-versatile',
    'Kimi-K2':         'moonshotai/kimi-k2-instruct-0905',
    'Qwen3-32B':       'qwen/qwen3-32b',
    'Llama4-Scout':    'meta-llama/llama-4-scout-17b-16e-instruct',
}

print(f'Groq clients: {len(groq_clients)} keys')
print(f'Modelos: {len(MODELS)}')
for name, model_id in MODELS.items():
    print(f'  {name}: {model_id}')

### **3. Prompt e Support Set**

In [ ]:
def build_few_shot_prompt(text, support_examples):
    """Few-shot: mostra N exemplos por classe antes de classificar."""
    examples_block = ''
    for _, row in support_examples.iterrows():
        ex_text = row['text'][:400]
        examples_block += f'Text: {ex_text}\nCategory: {row["label"]}\n\n'

    return f"""You are an expert at detecting AI-generated text. Classify the following text into exactly ONE of these categories:

- Human (written by a human, e.g. from Wikipedia)
- Anthropic (generated by Claude)
- Google (generated by Gemini)
- Meta (generated by Llama)
- OpenAI (generated by GPT)

Here are labeled examples:

{examples_block}
Now classify this text. Output ONLY the category name, nothing else.

Text: {text}
Category:"""


N_PER_CLASS = 30
support_set = pd.DataFrame()
for label in LABELS:
    subset = df_support[df_support['label'] == label]
    sampled = subset.sample(min(N_PER_CLASS, len(subset)), random_state=42)
    support_set = pd.concat([support_set, sampled], ignore_index=True)

print(f'Support set: {len(support_set)} exemplos')
for label in LABELS:
    count = len(support_set[support_set['label'] == label])
    print(f'  {label}: {count}')

### **4. Classificação via Groq (com rotação de keys e retries)**

In [ ]:
def normalize_prediction(raw):
    """Normaliza a resposta do LLM para um dos 5 labels."""
    if not raw or raw.startswith('Error'):
        return None
    raw_lower = raw.strip().lower()
    # Primeiro: match exato (a resposta é só o label)
    for label in LABELS:
        if raw_lower == label.lower():
            return label
    # Segundo: o label aparece algures na resposta
    for label in LABELS:
        if label.lower() in raw_lower:
            return label
    return None


def ask_groq(prompt, model_id, max_retries=4):
    """
    Chama o Groq com rotação automática de keys.
    Se uma key dá rate limit (429), tenta a próxima.
    """
    global groq_idx
    
    for attempt in range(max_retries):
        client = groq_clients[groq_idx % len(groq_clients)]
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=15,
                temperature=0.0,
            )
            groq_idx += 1  # Rodar para a próxima key no próximo pedido
            return response.choices[0].message.content.strip()
        except Exception as e:
            err_str = str(e)
            if '429' in err_str or 'rate_limit' in err_str.lower():
                groq_idx += 1  # Tentar próxima key
                time.sleep(2)
                continue
            else:
                return f'Error: {e}'
    
    return 'Error: rate_limit em todas as keys'


def run_groq_model(df_data, prompt_fn, support_df, model_name, model_id, sleep_seconds=0.5):
    """Corre um modelo Groq em todos os textos."""
    predictions = []
    raw_outputs = []
    errors = 0

    for _, row in tqdm(df_data.iterrows(), total=len(df_data), desc=model_name):
        text = row['text']
        try:
            prompt = prompt_fn(text, support_df)
            raw = ask_groq(prompt, model_id)
            pred = normalize_prediction(raw)
            if pred is None and raw and not raw.startswith('Error'):
                errors += 1
        except Exception as e:
            raw = f'Exception: {e}'
            pred = None

        raw_outputs.append(raw)
        predictions.append(pred)

        if sleep_seconds > 0:
            time.sleep(sleep_seconds)

    valid = sum(1 for p in predictions if p is not None)
    print(f'  → {valid}/{len(predictions)} válidas ({errors} respostas não-normalizáveis)')
    return predictions, raw_outputs


print('Funções prontas.')

### **5. Avaliação**

In [ ]:
def evaluate(predictions, gold_labels, title='', show_plot=True):
    """Avalia e mostra resultados. Retorna accuracy e F1 por classe."""
    valid = [(p, g) for p, g in zip(predictions, gold_labels) if p is not None]
    if not valid:
        print('Sem previsões válidas!')
        return 0.0, {}

    preds_clean, golds_clean = zip(*valid)
    acc = sum(p == g for p, g in valid) / len(valid)

    print(f'\n{"=" * 60}')
    print(f'{title} — Accuracy: {acc:.2%} ({len(valid)}/{len(predictions)} válidas)')
    print(f'{"=" * 60}')
    report = classification_report(golds_clean, preds_clean, labels=LABELS,
                                    zero_division=0, output_dict=True)
    print(classification_report(golds_clean, preds_clean, labels=LABELS, zero_division=0))

    f1_per_class = {label: report[label]['f1-score'] for label in LABELS}

    if show_plot:
        cm = confusion_matrix(golds_clean, preds_clean, labels=LABELS)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=LABELS, yticklabels=LABELS)
        plt.xlabel('Previsão'); plt.ylabel('Real')
        plt.title(f'{title} — {acc:.2%}')
        plt.tight_layout(); plt.show()

    return acc, f1_per_class

### **6. Correr Todos os Modelos**

In [ ]:
gold = df_test['label'].tolist()

# Guardar resultados de cada modelo
all_preds = {}   # {model_name: [preds]}
all_raw = {}     # {model_name: [raw_outputs]}
all_acc = {}     # {model_name: accuracy}
all_f1 = {}      # {model_name: {classe: f1}}

for model_name, model_id in MODELS.items():
    print(f'\n{"─" * 60}')
    print(f'🚀 A correr {model_name} ({model_id})')
    print(f'{"─" * 60}')
    
    preds, raw = run_groq_model(
        df_test, build_few_shot_prompt, support_set,
        model_name=model_name, model_id=model_id,
        sleep_seconds=0.5
    )
    
    acc, f1 = evaluate(preds, gold, model_name, show_plot=False)
    
    all_preds[model_name] = preds
    all_raw[model_name] = raw
    all_acc[model_name] = acc
    all_f1[model_name] = f1

print(f'\n{"═" * 60}')
print('TODOS OS MODELOS CONCLUÍDOS')
print(f'{"═" * 60}')
print(f'{"Modelo":<18} {"Accuracy":>10}')
print(f'{"-"*18} {"-"*10}')
for name in MODELS:
    print(f'{name:<18} {all_acc[name]:>10.2%}')

### **7. Confusion Matrices Individuais**

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, (model_name, preds) in enumerate(all_preds.items()):
    valid = [(p, g) for p, g in zip(preds, gold) if p is not None]
    if valid:
        p_clean, g_clean = zip(*valid)
        cm = confusion_matrix(g_clean, p_clean, labels=LABELS)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=LABELS, yticklabels=LABELS, ax=axes[idx])
        axes[idx].set_title(f'{model_name} — {all_acc[model_name]:.1%}', fontsize=11)
        axes[idx].set_xlabel('Previsão'); axes[idx].set_ylabel('Real')

# Esconder o 6º subplot (temos só 5 modelos)
axes[5].axis('off')

plt.suptitle('Confusion Matrices — Modelos Individuais', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### **8. Ensemble: Weighted Voting por F1/Classe**

Para cada texto, cada modelo vota na classe que previu.
O voto é ponderado pelo F1 que esse modelo obteve nessa classe.
A classe com maior peso total ganha.

In [ ]:
def weighted_voting_ensemble(predictions_dict, f1_dict, labels):
    """
    Ensemble por weighted voting usando F1 por classe como peso.
    
    Args:
        predictions_dict: {model_name: [preds]}
        f1_dict:          {model_name: {classe: f1}}
        labels:           lista de labels possíveis
    
    Returns:
        Lista de previsões finais, detalhes dos votos
    """
    model_names = list(predictions_dict.keys())
    n_samples = len(predictions_dict[model_names[0]])
    ensemble_preds = []
    vote_details = []
    
    for i in range(n_samples):
        weighted_scores = {label: 0.0 for label in labels}
        details = {}
        
        for model_name in model_names:
            pred = predictions_dict[model_name][i]
            if pred is not None and pred in f1_dict[model_name]:
                weight = f1_dict[model_name][pred]
                weighted_scores[pred] += weight
                details[model_name] = (pred, round(weight, 3))
        
        if max(weighted_scores.values()) == 0:
            # Fallback: primeira previsão não-None
            fallback = None
            for model_name in model_names:
                p = predictions_dict[model_name][i]
                if p is not None:
                    fallback = p
                    break
            ensemble_preds.append(fallback)
        else:
            best = max(weighted_scores, key=weighted_scores.get)
            ensemble_preds.append(best)
        
        vote_details.append((details, weighted_scores))
    
    return ensemble_preds, vote_details


# Também implementar majority voting simples para comparação
def majority_voting_ensemble(predictions_dict, labels):
    """Ensemble por majority voting simples (1 modelo = 1 voto)."""
    model_names = list(predictions_dict.keys())
    n_samples = len(predictions_dict[model_names[0]])
    ensemble_preds = []
    
    for i in range(n_samples):
        votes = []
        for model_name in model_names:
            pred = predictions_dict[model_name][i]
            if pred is not None:
                votes.append(pred)
        
        if votes:
            from collections import Counter
            counter = Counter(votes)
            ensemble_preds.append(counter.most_common(1)[0][0])
        else:
            ensemble_preds.append(None)
    
    return ensemble_preds


print('Funções de ensemble prontas.')

In [ ]:
# ============================================================
# TABELA DE PESOS (F1 por classe por modelo)
# ============================================================

print('Pesos do Ensemble (F1 por classe):')
print()
header = f'{"Classe":<12}'
for name in MODELS:
    header += f' {name:>14}'
print(header)
print('─' * len(header))

for label in LABELS:
    row = f'{label:<12}'
    values = []
    for name in MODELS:
        f1_val = all_f1[name].get(label, 0)
        values.append(f1_val)
        row += f' {f1_val:>14.3f}'
    # Marcar o melhor
    best_idx = np.argmax(values)
    print(row)
print()

# ============================================================
# WEIGHTED VOTING
# ============================================================
print('🏆 Ensemble — Weighted Voting')
weighted_preds, vote_details = weighted_voting_ensemble(all_preds, all_f1, LABELS)
weighted_acc, weighted_f1 = evaluate(weighted_preds, gold, 'Ensemble — Weighted Voting')

# ============================================================
# MAJORITY VOTING (para comparação)
# ============================================================
print('\n📊 Ensemble — Majority Voting (para comparação)')
majority_preds = majority_voting_ensemble(all_preds, LABELS)
majority_acc, majority_f1 = evaluate(majority_preds, gold, 'Ensemble — Majority Voting')

### **9. Comparação Final**

In [ ]:
print('\n' + '═' * 60)
print('RESUMO COMPARATIVO')
print('═' * 60)
for name in MODELS:
    print(f'  {name:<22} {all_acc[name]:>8.2%}')
print(f'  {"─"*22} {"─"*8}')
print(f'  {"Majority Voting":<22} {majority_acc:>8.2%}')
print(f'  {"Weighted Voting":<22} {weighted_acc:>8.2%}')
print()

best_solo = max(all_acc, key=all_acc.get)
best_solo_acc = all_acc[best_solo]
best_ensemble = max(weighted_acc, majority_acc)
gain = best_ensemble - best_solo_acc
print(f'Melhor modelo solo: {best_solo} ({best_solo_acc:.2%})')
print(f'Melhor ensemble:    {"Weighted" if weighted_acc >= majority_acc else "Majority"} ({best_ensemble:.2%})')
print(f'Ganho ensemble:     {gain:+.2%}')

# ============================================================
# GRÁFICO COMPARATIVO
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. Accuracy global
model_labels = list(MODELS.keys()) + ['Majority Vote', 'Weighted Vote']
accs = [all_acc[n] for n in MODELS] + [majority_acc, weighted_acc]
colors = ['#93c5fd', '#93c5fd', '#93c5fd', '#93c5fd', '#93c5fd', '#fcd34d', '#34d399']

bars = axes[0].barh(model_labels, accs, color=colors, height=0.6)
for bar, acc in zip(bars, accs):
    axes[0].text(acc + 0.005, bar.get_y() + bar.get_height()/2, 
                 f'{acc:.1%}', va='center', fontweight='bold', fontsize=10)
axes[0].set_xlim(0, 1.08)
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Accuracy Global', fontsize=13)

# 2. F1 por classe — ensemble vs melhor solo
x = np.arange(len(LABELS))
width = 0.3
axes[1].bar(x - width/2, [all_f1[best_solo].get(l, 0) for l in LABELS], 
            width, label=f'{best_solo} (melhor solo)', color='#93c5fd')
axes[1].bar(x + width/2, [weighted_f1.get(l, 0) for l in LABELS], 
            width, label='Weighted Ensemble', color='#34d399')
axes[1].set_xticks(x)
axes[1].set_xticklabels(LABELS, rotation=30, ha='right')
axes[1].set_ylabel('F1-Score')
axes[1].set_title('F1 por Classe: Melhor Solo vs Ensemble', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].set_ylim(0, 1.1)

plt.suptitle('Ensemble de 5 LLMs via Groq', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### **10. Análise de Erros e Concordância**

In [ ]:
# ============================================================
# CONCORDÂNCIA ENTRE MODELOS
# ============================================================
model_names = list(MODELS.keys())

print('Concordância entre modelos (% de textos com mesma previsão):')
print()
header = f'{"":>14}'
for n in model_names:
    header += f' {n:>14}'
print(header)

for n1 in model_names:
    row = f'{n1:>14}'
    for n2 in model_names:
        agree = sum(1 for a, b in zip(all_preds[n1], all_preds[n2]) 
                    if a == b and a is not None)
        total = sum(1 for a, b in zip(all_preds[n1], all_preds[n2]) 
                    if a is not None and b is not None)
        pct = agree / total if total > 0 else 0
        row += f' {pct:>14.1%}'
    print(row)

In [ ]:
# ============================================================
# ERROS DO ENSEMBLE WEIGHTED
# ============================================================

print('\n' + '═' * 60)
print('ERROS DO ENSEMBLE (Weighted Voting)')
print('═' * 60)

for i, (ens_pred, true) in enumerate(zip(weighted_preds, gold)):
    if ens_pred != true:
        texto_preview = df_test.iloc[i]['text'][:80]
        details, scores = vote_details[i]
        
        # Votos de cada modelo
        votes_str = ' | '.join([f'{m}: {p}({w})' for m, (p, w) in details.items()])
        
        print(f'  [{df_test.iloc[i]["id"]}] Real: {true} → Ensemble: {ens_pred}')
        print(f'    Votos: {votes_str}')
        print(f'    {texto_preview}...')
        print()

In [ ]:
# ============================================================
# ANÁLISE: QUANTOS ERROS O ENSEMBLE CORRIGIU/ESTRAGOU?
# ============================================================

print('\nImpacto do Ensemble vs cada modelo solo:')
print(f'{"Modelo":<18} {"Solo":>6} {"Ens. corrigiu":>14} {"Ens. estragou":>14}')
print(f'{"-"*18} {"-"*6} {"-"*14} {"-"*14}')

for name in MODELS:
    solo_correct = sum(1 for p, g in zip(all_preds[name], gold) if p == g)
    fixed = 0
    broke = 0
    for p_solo, p_ens, g in zip(all_preds[name], weighted_preds, gold):
        if p_solo != g and p_ens == g:
            fixed += 1
        if p_solo == g and p_ens != g:
            broke += 1
    print(f'{name:<18} {solo_correct:>6} {fixed:>+14} {broke:>14}')

### **11. Exportar Previsões**

In [ ]:
df_results = df_test[['id', 'label']].copy()

for name in MODELS:
    col_name = name.replace('-', '_').replace('.', '_').lower()
    df_results[col_name] = all_preds[name]

df_results['ensemble_majority'] = majority_preds
df_results['ensemble_weighted'] = weighted_preds
df_results['correct'] = df_results['label'] == df_results['ensemble_weighted']

df_results.to_csv('ensemble_groq_predictions.csv', index=False)
print(f'Previsões guardadas em ensemble_groq_predictions.csv')
print(f'\nPrimeiras linhas:')
df_results.head(10)